### 오답 분석 + 프롬프트 민감도
#### 오답 유형 분석
- 단순히 맞고 틀렸는지가 아니라 어떤 라벨을 어떤 라벨로 잘못 예측했는지 패턴을 찾음
    - RIGHT → PROC 로 오분류 3건   ← 가장 많은 혼동 패턴
    - CRIT  → ETC  로 오분류 2건
- 어떤 규칙을 프롬프트에 추가해야 하는지 근거가 생김

#### 프롬프트 민감도
- 같은 모델에 프롬프트만 바꿔서 성능 변화를 측정
    - zero-shot     → 정확도 40%
    - one-shot      → 정확도 60%
    - few-shot      → 정확도 80%
    - label-only    → 정확도 70%
    - rules-added   → 정확도 85%
- 변화가 크면 "병목은 모델 크기가 아니라 프롬프트 설계에있다"는 것이 증명

#### 출력 안정성
- 같은 문장을 여러 번 질의했을 때 라벨이 바뀌는지 확인
    - 1번째 질의: PROC
    - 2번째 질의: PROC
    - 3번째 질의: RIGHT  ← 흔들림 발생!
- 출력이 흔들리면 모델 이전에 샘플링 설정, 출력 형식, 파싱 규칙을 먼저 점검해야 한다

In [2]:
import json
import os
from collections import Counter, defaultdict
# defaultdict : 없는 키에 접근 할 때 자동으로 기본값을 생성
import numpy as np
import pandas as pd
from dotenv import load_dotenv

load_dotenv(override=True)
# override=True : 이미 설정된 환경변수도 .env값으로 덮으씀

True

In [12]:
BASE_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
LABELS     = ['DEF', 'RIGHT', 'PROC', 'ORG', 'CRIT', 'ETC']
LABEL_DESC = {
    'DEF':   '정의/목적/적용범위 조항',
    'RIGHT': '권리/의무/금지/책임 조항',
    'PROC':  '신청/심사/조사/불복/처벌 절차 조항',
    'ORG':   '기관/위원회/법원 등 조직의 설치/구성/권한 조항',
    'CRIT':  '자격/요건/기준/기간/수치 조건 조항',
    'ETC':   '시행일/경과조치/위임 등 기타 조항',
}
# 순서가 고정된 리스트로 따로 정의
# 혼동 행렬을 만들 때 항상 같은 순서로 행과 열을 배치해야 하기 때문

In [13]:
sample_data = [
    {'id': 'D01', 'category': 'DEF',   'text': '이 법은 국민의 기본적 인권을 보호하고 자유와 평등을 실현함을 목적으로 한다.'},
    {'id': 'D02', 'category': 'DEF',   'text': '이 법에서 사용하는 용어의 뜻은 다음 각 호와 같다.'},
    {'id': 'D03', 'category': 'DEF',   'text': '공공기관이란 국가기관, 지방자치단체 및 법령에 따라 설치된 기관을 말한다.'},
    {'id': 'R01', 'category': 'RIGHT', 'text': '모든 국민은 법 앞에 평등하며 성별, 종교 또는 사회적 신분에 의하여 차별을 받지 아니한다.'},
    {'id': 'R02', 'category': 'RIGHT', 'text': '사업자는 이용자의 개인정보를 안전하게 관리하여야 한다.'},
    {'id': 'P01', 'category': 'PROC',  'text': '이 법을 위반한 자는 3년 이하의 징역 또는 3천만원 이하의 벌금에 처한다.'},
    {'id': 'P02', 'category': 'PROC',  'text': '신청인은 처분 통지를 받은 날부터 30일 이내에 이의신청을 할 수 있다.'},
    {'id': 'O01', 'category': 'ORG',   'text': '분쟁 조정을 위하여 국무총리 소속으로 조정위원회를 둔다.'},
    {'id': 'O02', 'category': 'ORG',   'text': '위원회는 위원장 1명을 포함한 15명 이내의 위원으로 구성한다.'},
    {'id': 'C01', 'category': 'CRIT',  'text': '후보자는 선거일 현재 25세 이상인 국민이어야 한다.'},
    {'id': 'C02', 'category': 'CRIT',  'text': '허가를 받으려는 자는 자본금 1억원 이상과 전담 인력 2명 이상을 갖추어야 한다.'},
    {'id': 'E01', 'category': 'ETC',   'text': '이 법은 공포 후 6개월이 경과한 날부터 시행한다.'},
    {'id': 'E02', 'category': 'ETC',   'text': '이 법 시행 당시 종전의 규정에 따라 한 처분은 이 법에 따른 처분으로 본다.'},
]

sample_data += [
    {'id': 'R03', 'category': 'RIGHT', 'text': '누구든지 정당한 사유 없이 타인의 개인정보를 수집·이용 또는 제공하여서는 아니 된다.'},
    {'id': 'R04', 'category': 'RIGHT', 'text': '사용자는 근로자에게 최저임금 이상을 지급하여야 한다.'},
    {'id': 'R05', 'category': 'RIGHT', 'text': '행정기관은 민원인의 권익을 보호하기 위하여 필요한 조치를 하여야 한다.'},
    {'id': 'R06', 'category': 'RIGHT', 'text': '모든 국민은 건강하고 쾌적한 환경에서 생활할 권리를 가진다.'},
    {'id': 'R07', 'category': 'RIGHT', 'text': '사업주는 산업재해를 예방하기 위하여 안전 및 보건 조치를 이행하여야 한다.'},
    {'id': 'R08', 'category': 'RIGHT', 'text': '누구든지 폭행, 협박 또는 위계로 타인의 업무를 방해하여서는 아니 된다.'},
    {'id': 'R09', 'category': 'RIGHT', 'text': '국가기관은 법령에서 정한 경우를 제외하고는 개인의 자유와 권리를 제한하여서는 아니 된다.'},
    {'id': 'R10', 'category': 'RIGHT', 'text': '이용자는 자신의 개인정보 처리에 관한 사항을 열람하거나 정정을 요구할 수 있다.'},
    {'id': 'R11', 'category': 'RIGHT', 'text': '공무원은 직무상 알게 된 비밀을 누설하여서는 아니 된다.'},
    {'id': 'R12', 'category': 'RIGHT', 'text': '모든 국민은 법률이 정하는 바에 따라 납세의 의무를 진다.'},
    {'id': 'R13', 'category': 'RIGHT', 'text': '판매업자는 소비자에게 상품의 원산지와 가격 정보를 명확히 표시하여야 한다.'},
    {'id': 'R14', 'category': 'RIGHT', 'text': '아동은 어떠한 형태의 학대와 방임으로부터 보호받을 권리를 가진다.'},
    {'id': 'R15', 'category': 'RIGHT', 'text': '정보통신서비스 제공자는 이용자의 동의 없이 개인정보를 제3자에게 제공하여서는 아니 된다.'},
    {'id': 'R16', 'category': 'RIGHT', 'text': '국가와 지방자치단체는 장애인의 이동권 보장을 위하여 필요한 시책을 마련하여야 한다.'},
    {'id': 'R17', 'category': 'RIGHT', 'text': '누구든지 허위 또는 과장된 표시·광고를 하여 소비자를 기만하여서는 아니 된다.'},
    {'id': 'R18', 'category': 'RIGHT', 'text': '피해자는 범죄로 인한 손해의 배상을 청구할 권리를 가진다.'},
    {'id': 'R19', 'category': 'RIGHT', 'text': '사업자는 이용자의 서비스 이용기록을 안전하게 보관하여야 한다.'},
    {'id': 'R20', 'category': 'RIGHT', 'text': '모든 국민은 인간으로서의 존엄과 가치를 가지며 행복을 추구할 권리를 가진다.'},
]
# RIGHT를 의도적으로 늘림
# 모델이 가장 취약한 카테고리에 다양한 표현을 집중 보강 
# 모델이 특정 표현 하나만 RIGHT로 인식하는 편향을 줄이기 위해

df_all = pd.DataFrame(sample_data)
df     = df_all.copy()

print(f'평가 데이터 로드 완료: {len(df)}건')
df.head()

평가 데이터 로드 완료: 31건


,id,category,text
0,D01,DEF,이 법은 국민의 기본적 인권을 보호하고 자유와 평등을 실현함을 목적으로 한다.
1,D02,DEF,이 법에서 사용하는 용어의 뜻은 다음 각 호와 같다.
2,D03,DEF,"공공기관이란 국가기관, 지방자치단체 및 법령에 따라 설치된 기관을 말한다."
3,R01,RIGHT,"모든 국민은 법 앞에 평등하며 성별, 종교 또는 사회적 신분에 의하여 차별을 받지 ..."
4,R02,RIGHT,사업자는 이용자의 개인정보를 안전하게 관리하여야 한다.


In [14]:
# 시스템 프롬프트 + 프롬프트 빌더
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

SYSTEM_PROMPT = (
    '너는 한국 법률 조항 분류기다. 반드시 다음 6개 코드 중 하나만 선택하라: '
    + ', '.join(LABELS) + '. '
    '응답은 반드시 JSON 한 줄로만 반환한다. 형식: {"label":"DEF","reason":"..."}'
)

LABEL_GUIDE = '\n'.join([f'- {label}: {description}' for label, description in LABEL_DESC.items()])
# 딕셔너리를 순회해서 프롬프트에 바로 붙여넣을 수 있는 문자열로 변환
# - DEF: 정의/목적/적용범위 조항
# - RIGHT: 권리/의무/금지/책임 조항
# - PROC: 신청/심사/조사/불복/처벌 절차 조항
# - ORG: 기관/위원회/법원 등 조직의 설치/구성/권한 조항
# - CRIT: 자격/요건/기준/기간/수치 조건 조항
# - ETC: 시행일/경과조치/위임 등 기타 조항

def build_prompt(text, mode='zero', examples=None, rules=None, label_only=False):
    # rules=True : 구분 규칙을 동적으로 주입할 수 있음
    # label_only=True : reason없이 label만 출력하게 강제 
    base = [
        '다음 법률 문장을 6개 코드 중 하나로 분류하라.',
        '라벨 설명:',
        LABEL_GUIDE,
    ]
    if rules:
        base += ['', '구분 규칙:']
        base += [f'- {rule}' for rule in rules]
    if mode == 'one' and examples:
        example = examples[0]
        base += [
            '',
            '예시 1개:',
            f"문장: {example['text']}",
            '정답(JSON): ' + json.dumps(example['output'], ensure_ascii=False),
        ]
    elif mode == 'few' and examples:
        base += ['', f'예시 {len(examples)}개:']
        for index, example in enumerate(examples, 1):
            base += [
                f'예시 {index}:',
                f"문장: {example['text']}",
                '정답(JSON): ' + json.dumps(example['output'], ensure_ascii=False),
            ]
    base += ['', f'분류할 문장: {text}']
    if label_only:
        base += ['JSON은 label만 포함하라.', '{"label":"DEF|RIGHT|PROC|ORG|CRIT|ETC"}']
    else:
        base += ['JSON으로만 응답하라.', '{"label":"DEF|RIGHT|PROC|ORG|CRIT|ETC","reason":"짧은 근거"}']
    return '\n'.join(base)

print(build_prompt(df['text'][0]))

다음 법률 문장을 6개 코드 중 하나로 분류하라.
라벨 설명:
- DEF: 정의/목적/적용범위 조항
- RIGHT: 권리/의무/금지/책임 조항
- PROC: 신청/심사/조사/불복/처벌 절차 조항
- ORG: 기관/위원회/법원 등 조직의 설치/구성/권한 조항
- CRIT: 자격/요건/기준/기간/수치 조건 조항
- ETC: 시행일/경과조치/위임 등 기타 조항

분류할 문장: 이 법은 국민의 기본적 인권을 보호하고 자유와 평등을 실현함을 목적으로 한다.
JSON으로만 응답하라.
{"label":"DEF|RIGHT|PROC|ORG|CRIT|ETC","reason":"짧은 근거"}


In [15]:
# 추론 함수 정의
def load_hf_model(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model     = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype='auto', device_map='auto'
    )
    return model, tokenizer

def extract_json(text):
    start = text.find('{')
    end   = text.find('}')
    if start == -1 or end == -1 or end <= start:
        return None
    try:
        return json.loads(text[start:end+1])
    except json.JSONDecodeError:
        return None

def predict(model, tokenizer, prompt,
            max_new_tokens=128, do_sample=False,
            temperature=None, top_p=None):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': prompt},
    ]
    text_input   = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    model_inputs = tokenizer([text_input], return_tensors='pt').to(model.device)

    generate_kwargs = {
        'max_new_tokens': max_new_tokens,
        'do_sample':      do_sample,
        'pad_token_id':   tokenizer.eos_token_id,
    }
    if temperature is not None:
        generate_kwargs['temperature'] = temperature
    if top_p is not None:
        generate_kwargs['top_p'] = top_p

    generated_ids = model.generate(**model_inputs, **generate_kwargs)
    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    raw_response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    parsed = extract_json(raw_response)
    return {
        'raw_response': raw_response,
        'label':  parsed.get('label')  if parsed else 'PARSE_FAIL',
        'reason': parsed.get('reason') if parsed else raw_response[:160],
    }

def normalize_prediction(label):
    if isinstance(label, str) and label in LABELS:
        return label
    return 'PARSE_FAIL'

print('프롬프트/추론 함수 정의 완료')

프롬프트/추론 함수 정의 완료


In [16]:
# load_hf_model
# 모델과 토크나이저를 한 번에 로드해서 반환하는 헬퍼 함수.
# 나중에 베이스 모델과 LoRA튜닝 모델을 번갈아 로드할 때 코드가 깔끔해짐
def load_hf_model(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model     = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype='auto', device_map='auto'
    )
    return model, tokenizer

In [17]:
BASE_ZERO_FEW_EXAMPLES = [
    {
        'text':   '신청인은 처분 통지를 받은 날부터 30일 이내에 이의신청을 할 수 있다',
        'output': {'label': 'PROC', 'reason': '불복 절차를 규정한다'}
    }
]
# one-shot / few-shot 모드에서 공통으로 사용할 예시
# 모든 실험에서 동일한 예시를 써야 공정한 비교 가능

def run_eval(model, tokenizer, mode='zero', label_only=False,
             rules=None, predict_kwargs=None):
    rows = []
    for sample in df.to_dict('records'):
        prompt = build_prompt(
            sample['text'],
            mode     = mode,
            examples = BASE_ZERO_FEW_EXAMPLES if mode in ('one', 'few') else None,
            rules    = rules,
            label_only = label_only,
        )
        predict_args = predict_kwargs or {}
        result = predict(model, tokenizer, prompt,
                         max_new_tokens=128, **predict_args)
        rows.append({
            'id':               sample['id'],
            'true_label':       sample['category'],
            'predicted_label':  normalize_prediction(result['label']),
            'reason':           result['reason'],
            'raw_response':     result['raw_response'],
            'correct':          normalize_prediction(result['label']) == sample['category'],
            'text':             sample['text'],
        })
    return pd.DataFrame(rows)

base_model, base_tokenizer = load_hf_model(BASE_MODEL)
base_pred_df = run_eval(base_model, base_tokenizer)
base_pred_df

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 5462.78it/s]


,id,true_label,predicted_label,reason,raw_response,correct,text
0,D01,DEF,DEF,국민의 기본적 인권을 보호하고 자유와 평등을 실현함을 목적으로 하는 법입니다.,"{""label"":""DEF"", ""reason"":""국민의 기본적 인권을 보호하고 자유와...",True,이 법은 국민의 기본적 인권을 보호하고 자유와 평등을 실현함을 목적으로 한다.
1,D02,DEF,DEF,정의/목적/적용범위,"{""label"":""DEF"", ""reason"":""정의/목적/적용범위""}",True,이 법에서 사용하는 용어의 뜻은 다음 각 호와 같다.
2,D03,DEF,DEF,"국가기관, 지방자치단체 및 법령에 따라 설치된 기관","{""label"":""DEF"", ""reason"":""국가기관, 지방자치단체 및 법령에 따...",True,"공공기관이란 국가기관, 지방자치단체 및 법령에 따라 설치된 기관을 말한다."
3,R01,RIGHT,PARSE_FAIL,문장 내용은 법적 제도를 적용하는 것에 대한 명시적인 조항입니다.,"{""label"":""DEF|RIGHT|PROC|ORG|CRIT|ETC"",""reason...",False,"모든 국민은 법 앞에 평등하며 성별, 종교 또는 사회적 신분에 의하여 차별을 받지 ..."
4,R02,RIGHT,PARSE_FAIL,작업자와 이용자가 개인정보를 안전하게 관리해야 합니다.,"{""label"":""DEF|RIGHT|PROC|ORG|CRIT|ETC"",""reason...",False,사업자는 이용자의 개인정보를 안전하게 관리하여야 한다.
5,P01,PROC,DEF,정의/목적/적용범위,"{""label"":""DEF"", ""reason"":""정의/목적/적용범위""}",False,이 법을 위반한 자는 3년 이하의 징역 또는 3천만원 이하의 벌금에 처한다.
6,P02,PROC,DEF,30일 이내에 이의신청을 할 수 있다,"{""label"":""DEF"", ""reason"":""30일 이내에 이의신청을 할 수 있다""}",False,신청인은 처분 통지를 받은 날부터 30일 이내에 이의신청을 할 수 있다.
7,O01,ORG,DEF,국무총리는 분쟁 조정을 위한 조정위원회를 설립해야 합니다.,"{""label"":""DEF"", ""reason"":""국무총리는 분쟁 조정을 위한 조정위원...",False,분쟁 조정을 위하여 국무총리 소속으로 조정위원회를 둔다.
8,O02,ORG,DEF,정의/목적/적용범위,"{""label"":""DEF"", ""reason"":""정의/목적/적용범위""}",False,위원회는 위원장 1명을 포함한 15명 이내의 위원으로 구성한다.
9,C01,CRIT,DEF,후보자가 선거일 현재 25세 이상인 국민이야,"{""label"":""DEF"", ""reason"":""후보자가 선거일 현재 25세 이상인 ...",False,후보자는 선거일 현재 25세 이상인 국민이어야 한다.


In [18]:
def metric_summary(pred_df):
    accuracy    = float(pred_df['correct'].mean())
    labels      = sorted(df['category'].unique())
    precisions, recalls, f1s = [], [], []

    for label in labels:
        tp = int(((pred_df['true_label'] == label) & (pred_df['predicted_label'] == label)).sum())
        fp = int(((pred_df['true_label'] != label) & (pred_df['predicted_label'] == label)).sum())
        fn = int(((pred_df['true_label'] == label) & (pred_df['predicted_label'] != label)).sum())
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall    = tp / (tp + fn) if (tp + fn) else 0.0
        f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
        precisions.append(precision)
        recalls.append(recall)
        f1s.append(f1)

    return {
        'accuracy':         accuracy,
        'macro_precision':  float(np.mean(precisions)),
        'macro_recall':     float(np.mean(recalls)),
        'macro_f1':         float(np.mean(f1s)),
    }

metric_summary(base_pred_df)

{'accuracy': 0.0967741935483871,
 'macro_precision': 0.027777777777777776,
 'macro_recall': 0.16666666666666666,
 'macro_f1': 0.047619047619047616}

In [19]:
label_order    = LABELS
label_to_index = {label: index for index, label in enumerate(label_order)}

def confusion_matrix_from_df(pred_df):
    matrix = np.zeros((len(label_order), len(label_order)), dtype=int)
    for _, row in pred_df.iterrows():
        true_label = row['true_label']
        pred_label = row['predicted_label']
        if true_label in label_to_index and pred_label in label_to_index:
            matrix[label_to_index[true_label], label_to_index[pred_label]] += 1
    return matrix

def per_label_metrics(pred_df):
    rows = []
    for label in label_order:
        tp = int(((pred_df['true_label'] == label) & (pred_df['predicted_label'] == label)).sum())
        fp = int(((pred_df['true_label'] != label) & (pred_df['predicted_label'] == label)).sum())
        fn = int(((pred_df['true_label'] == label) & (pred_df['predicted_label'] != label)).sum())
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall    = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
        rows.append({
            'label':     label,
            'precision': precision,
            'recall':    recall,
            'f1':        f1,
            'support':   int((pred_df['true_label'] == label).sum()),
        })
    return pd.DataFrame(rows)

confusion    = confusion_matrix_from_df(base_pred_df)
label_metrics = per_label_metrics(base_pred_df)

print('혼동행렬')
pd.DataFrame(confusion, index=label_order, columns=label_order)

print('라벨별 지표')
print(label_metrics)

혼동행렬
라벨별 지표
   label  precision  recall        f1  support
0    DEF   0.166667     1.0  0.285714        3
1  RIGHT   0.000000     0.0  0.000000       20
2   PROC   0.000000     0.0  0.000000        2
3    ORG   0.000000     0.0  0.000000        2
4   CRIT   0.000000     0.0  0.000000        2
5    ETC   0.000000     0.0  0.000000        2


In [20]:
wrong_df = base_pred_df[base_pred_df['correct'] == False].copy()

wrong_df['error_type'] = wrong_df.apply(
    lambda row: f"{row['true_label']}->{row['predicted_label']}", axis=1
)

error_group = (
    wrong_df
    .groupby('error_type')
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)

print('오답유형')
print(error_group)

오답유형
          error_type  count
6  RIGHT->PARSE_FAIL     12
5         RIGHT->DEF      8
2           ETC->DEF      2
4          PROC->DEF      2
3           ORG->DEF      2
1   CRIT->PARSE_FAIL      1
0          CRIT->DEF      1


In [ ]:
prompt_variants = [
    ('zero',       {'mode': 'zero',  'label_only': False, 'rules': None}),
    ('one',        {'mode': 'one',   'label_only': False, 'rules': None}),
    ('few',        {'mode': 'few',   'label_only': False, 'rules': None}),
    ('label_only', {'mode': 'zero',  'label_only': True,  'rules': None}),
    ('rules_added',{'mode': 'zero',  'label_only': False, 'rules': [
        'DEF는 의무, 금지, 책임처럼 행위 기준을 직접 제시하는 문장이다.',
        'RIGHT는 권리, 청구, 자격처럼 주체의 권한을 보장하는 문장이다.',
        'PROC는 신청, 신고, 기간, 절차처럼 진행 과정을 말한다.',
    ]}),
    ('sample_low_temp',  {'mode': 'zero', 'label_only': False, 'rules': None,
                          'predict_kwargs': {'do_sample': True, 'temperature': 0.2}}),
    ('sample_high_temp', {'mode': 'zero', 'label_only': False, 'rules': None,
                          'predict_kwargs': {'do_sample': True, 'temperature': 0.8}}),
]

prompt_sensitivity_rows = []
for variant_name, config in prompt_variants:
    pred_variant_df = run_eval(
        base_model, base_tokenizer,
        mode         = config['mode'],
        label_only   = config['label_only'],
        rules        = config['rules'],
        predict_kwargs = config.get('predict_kwargs'),
    )
    summary = metric_summary(pred_variant_df)
    prompt_sensitivity_rows.append({'variant': variant_name, **summary})

prompt_sensitivity_df = pd.DataFrame(prompt_sensitivity_rows)
print('프롬프트 민감도 비교')
print(prompt_sensitivity_df)

In [ ]:
stability_cases = df.head(5).to_dict('records')
stability_rows  = []

for sample in stability_cases:
    labels_seen = []
    for _ in range(3):    # 같은 문장을 3번 질의
        result = predict(
            base_model, base_tokenizer,
            build_prompt(sample['text']),
            max_new_tokens=128,
            do_sample=False    # deterministic
        )
        labels_seen.append(normalize_prediction(result['label']))

    stability_rows.append({
        'id':          sample['id'],
        'true_label':  sample['category'],
        'labels_seen': labels_seen,
        'stable':      len(set(labels_seen)) == 1,   # 3번 모두 같으면 True
    })

stability_df = pd.DataFrame(stability_rows)
print('출력 안정성 샘플')
print(stability_df)

### 학습 데이터 자동 생성
- 직접 만드는 이유
    - 저작권/개인정보 문제
    - 라벨링 비용 (사람이 직접 분류해야 함)
- 템플릿 기반으로 가상의 법률 문장을 자동 생성
- 완벽하진 않지만 LoRA 튜닝에 필요한 최소한의 패턴을 학습시킬 수 있음

In [ ]:
import json
import random
from pathlib import Path

TEMPLATES = {
    'DEF': [
        '이 규정에서 사용되는 용어의 정의는 다음과 같다: {term}은 {definition}을 말한다.',
        '{term}의 목적은 {purpose}에 있으며, 적용범위는 {scope}로 한다.',
    ],
    'RIGHT': [
        '사용자는 {action}할 권리를 가지며, 다음과 같은 의무를 진다: {duty}.',
        '사업자는 {prohibition}을 금지하며, 위반 시 {penalty}의 책임을 진다.',
    ],
    'PROC': [
        '신청은 {who}에게 제출하며, 심사는 {period} 이내에 완료한다.',
        '불복 절차는 {procedure}에 따라 진행되며, 이의신청은 {deadline}까지 가능하다.',
    ],
    'ORG': [
        '위원회는 {members}로 구성되며, 위원회의 권한은 {authority}로 한다.',
        '{org}를 설치하여 {task}를 수행한다.',
    ],
    'CRIT': [
        '응시자는 {qualification}을 갖추어야 하며, 자격 유효기간은 {period}이다.',
        '선정 기준은 {criteria}이며, 최소 점수는 {score} 이상으로 한다.',
    ],
    'ETC': [
        '이 규정은 {effective_date}부터 시행한다.',
        '경과조치는 {transition}에 따른다.',
    ],
}

In [ ]:
def synthesize_clause(label):
    tpl = random.choice(TEMPLATES[label])   # 템플릿 랜덤 선택
    while '{' in tpl:                        # 플레이스홀더가 없을 때까지 반복
        start = tpl.find('{')
        end   = tpl.find('}', start)
        key   = tpl[start+1:end]             # 키워드 추출
        tpl   = tpl[:start] + random_phrase(key) + tpl[end+1:]  # 랜덤 표현으로 교체
    return tpl

In [ ]:
def make_prompt(clause):
    labels  = ', '.join([f"{k}({v})" for k, v in LABEL_DESC.items()])
    prompt  = (
        f"다음 조항을 다음 라벨 중 하나로 분류하시오. 가능한 라벨: {labels}\n"
        f"조항: {clause}\n라벨:"
    )
    return prompt

In [ ]:
def generate(out_dir='./finetune_data', train_size=2000, val_size=400, seed=42):
    random.seed(seed)
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)

    def write_file(path, n):
        with open(path, 'w', encoding='utf-8') as f:
            for _ in range(n):
                label  = random.choice(list(LABEL_DESC.keys()))
                clause = synthesize_clause(label)
                prompt = make_prompt(clause)
                item   = {'prompt': prompt, 'response': label}
                f.write(json.dumps(item, ensure_ascii=False) + '\n')

    write_file(out / 'train.jsonl',      train_size)
    write_file(out / 'validation.jsonl', val_size)
    print(f'Wrote train/validation to {out.resolve()}')

if __name__ == '__main__':
    generate()

### LoRA Fine-tuning
#### LoRA (Low Rank Adaptation)
- 모델 전체를 학습하는 Full Fine-tuning 대신 소수의 추가 파라미터만 학습
- Full Fine-tuning vs LoRA:
    - Full Fine-tuning:
    - 모든 가중치 W (5억개) 전부 업데이트
    - → 메모리, 시간, 비용 매우 큼

    - LoRA:
    - 기존 가중치 W는 그대로 고정
    - 작은 행렬 A, B (440만개)만 새로 학습
    = → 전체의 0.88%만 학습!

#### LoRA 수학적 원리
- 기존 가중치: W (큰 행렬, 고정)
- LoRA 추가: ΔW = A × B (작은 두 행렬의 곱)
- 실제 연산: output = W × input + (A × B) × input
    - A : (d × r) 크기, r 은 rank (보통 4~16)
    - B : (r × d) 크기
    - r 이 작을수록 파라미터 수가 줄어들지만 표현력도 줄어듦

#### LoRA 적용
- Transformer의 선형층(Linear Layer)에 적용
    - Attention 모듈의 선형층:
    - q_proj, k_proj, v_proj, o_proj  ← Q/K/V/Output 변환

    - MLP 모듈의 선형층:
    - gate_proj, up_proj, down_proj   ← Feed-Forward Network

In [ ]:
import torch
import torch.nn as nn
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# ============================================================
# STEP 1. 설정값
# ============================================================
model_name                  = 'Qwen/Qwen2.5-0.5B-Instruct'
train_file                  = r'finetune_data\train.jsonl'
valid_file                  = r'finetune_data\validation.jsonl'
output_dir                  = 'peft_output'
num_train_epochs            = 3
per_device_train_batch_size = 4
learning_rate               = 2e-4

if torch.cuda.is_available():
    use_bf16 = True
    use_fp16 = False
else:
    use_bf16 = False
    use_fp16 = True

lora_r       = 8
lora_alpha   = 16
lora_dropout = 0.05
use_4bit     = False
max_leng     = 512

LABEL_DESC = {
    'DEF':   '정의/목적/적용범위 조항',
    'RIGHT': '권리/의무/금지/책임 조항',
    'PROC':  '신청/심사/조사/불복/처벌 절차 조항',
    'ORG':   '기관/위원회/법원 등 조직의 설치/구성/권한 조항',
    'CRIT':  '자격/요건/기준/기간/수치 조건 조항',
    'ETC':   '시행일/경과조치/위임 등 기타 조항',
}
print('Config set:', model_name)

# ============================================================
# STEP 2. 전처리 함수
# ============================================================
def build_inputs_and_labels(batch, tokenizer, max_length=512):
    inputs = []
    labels = []
    for p, r in zip(batch['prompt'], batch['response']):
        full             = p + ' ' + r
        tokenized_full   = tokenizer(full, truncation=True, max_length=max_length)
        tokenized_prompt = tokenizer(p,    truncation=True, max_length=max_length)
        input_ids        = tokenized_full['input_ids']
        label_ids        = input_ids.copy()
        prompt_len       = len(tokenized_prompt['input_ids'])
        for i in range(min(prompt_len, len(label_ids))):
            label_ids[i] = -100
        inputs.append(input_ids)
        labels.append(label_ids)
    return {'input_ids': inputs, 'labels': labels}

# ============================================================
# STEP 3. 토크나이저 + 모델 로드
# ============================================================
tokenizer           = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
load_kwargs         = {'trust_remote_code': True}

if use_4bit:
    load_kwargs.update({'load_in_4bit': True, 'device_map': 'auto'})

model = AutoModelForCausalLM.from_pretrained(model_name, **load_kwargs)

if use_4bit:
    model = prepare_model_for_kbit_training(model)

print('model load')

# ============================================================
# STEP 4. LoRA 타겟 모듈 자동 탐색 + 적용
# ============================================================
def find_lora_target_modules(model):
    linear_leaf_names = set()
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            linear_leaf_names.add(name.split('.')[-1])
    preferred = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
    selected  = [m for m in preferred if m in linear_leaf_names]
    if not selected:
        fallback_keywords = ('proj', 'wq', 'wk', 'wv', 'wo', 'fc')
        selected = sorted([
            n for n in linear_leaf_names
            if any(k in n for k in fallback_keywords) and n not in {'lm_head'}
        ])
    if not selected:
        raise ValueError(f'LoRA Target Module를 자동 탐색하지 못했습니다. 발견된 linear leaf name: {linear_leaf_names}')
    return selected, sorted(linear_leaf_names)

target_modules, linear_names = find_lora_target_modules(model)
print('Detected linear layer names:', linear_names[:30])
print('Selected target_modules for LoRA:', target_modules)

peft_config = LoraConfig(
    r              = lora_r,
    lora_alpha     = lora_alpha,
    target_modules = target_modules,
    lora_dropout   = lora_dropout,
    bias           = 'none',
    task_type      = 'CAUSAL_LM',
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# ============================================================
# STEP 5. 데이터셋 로드 + 토크나이징
# ============================================================
data_files      = {'train': train_file, 'validation': valid_file}
dataset         = load_dataset('json', data_files=data_files)

def preprocess_batch(batch):
    return build_inputs_and_labels(batch, tokenizer, max_length=max_leng)

tokenized_train = dataset['train'].map(
    lambda x: preprocess_batch(x),
    batched=True,
    remove_columns=dataset['train'].column_names
)
tokenized_eval  = dataset['validation'].map(
    lambda x: preprocess_batch(x),
    batched=True,
    remove_columns=dataset['validation'].column_names
)

def collate_with_padding(features):
    input_ids = [f['input_ids'] for f in features]
    labels    = [f['labels']    for f in features]
    padded    = tokenizer.pad({'input_ids': input_ids}, padding=True, return_tensors='pt')
    max_len   = padded['input_ids'].shape[1]
    padded_labels = []
    for lb in labels:
        padded_labels.append(lb + [-100] * (max_len - len(lb)))
    padded['labels'] = torch.tensor(padded_labels, dtype=torch.long)
    return padded

print('Datasets prepared:', len(tokenized_train), len(tokenized_eval))

# ============================================================
# STEP 6. Trainer 구성 + 학습
# ============================================================
base_kwargs = dict(
    output_dir                  = output_dir,
    per_device_train_batch_size = per_device_train_batch_size,
    per_device_eval_batch_size  = per_device_train_batch_size,
    num_train_epochs            = num_train_epochs,
    learning_rate               = learning_rate,
    bf16                        = use_bf16,
    fp16                        = use_fp16,
    save_total_limit            = 3,
    remove_unused_columns       = False,
    logging_steps               = 10,
)

try:
    training_args = TrainingArguments(**{**base_kwargs, 'evaluation_strategy': 'epoch'})
except TypeError:
    training_args = TrainingArguments(**base_kwargs)

trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = tokenized_train,
    eval_dataset  = tokenized_eval,
    data_collator = collate_with_padding
)

train_result = trainer.train()
print(f'loss: {getattr(train_result, "training_loss", None)}')